Next steps: 
- test on variety of adatas
- make sure it works with csvs too 



In [ ]:
import numpy as np
from matplotlib import pyplot as plt


from scphere.model.vae import SCPHERE

import pandas as pd
import anndata as ad

import plotly.graph_objects as go
import plotly.express as px

from scipy.sparse import issparse
from scphere.util.trainer import Trainer

import sys
sys.path.append('/projects/steiflab/research/hmacdonald/total_RNA/Ouroboros_pypi/ouroboros')

from ouroboros_functions import *


/projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future versi

In [ ]:


def run_ouroboros(data, data_type, species = 'human', outdir = '.'):

    if data_type == 'h5ad':
        data = ad.read_h5ad(data)
        data.X = data.layers['raw_counts'].copy()
        
    elif data_type == 'csv':
        data = pd.read_csv(data)
        data = data.set_index('cell_id')
    else: 
        raise TypeError("Unsupported data type. Expected --h5ad or --csv for data_type.")

        
    if species == 'mouse':
        print('Converting mouse genes to human orthologs...')
        data = convert_to_human_genes(data)
        if isinstance(data, ad.AnnData):
            data.layers['raw_counts'] = data.X.copy()
        print('Genes successfully converted to human orthologs')
    elif species == 'human':
        pass
    else:
        raise TypeError("Unsupported species. Model only optimized for --human or --mouse")

    missing = check_features(data)

    if len(missing) > 0:
        print(f"""Key training genes seem to be missing from your dataset\n
              Missing genes include: {missing}
              For higher accuracy consider including these genes in the matrix and running Ouroboros again.
              Retraining model without them......""")
        model, ref_embed, in_order_feature_set = ouroboros_retrain(data)
        ref_embed.to_csv(f'{outdir}/retrained_reference_embeddings.csv')
        z_df = embed_in_retrained_sphere(data, model, in_order_feature_set)
        z_df = KNN_predict(ref_embed, z_df)
        pseud, ref_pseud = dormancy_depth(z_df, ref_embed, retrained = True)
        z_df = z_df.merge(pseud, how = 'left', left_index = True, right_index = True)
        z_df.to_csv(f'{outdir}/ouroboros_embeddings_pseudotimes.csv')
        return z_df
        
    else:
        print('making matrix')
        matrix = ouroboros_preprocess(data, data_type, species = 'human')
        z_df = ouroboros_embed(matrix, data, data_type, outdir = outdir)
        z_df.to_csv(f'{outdir}/ouroboros_embeddings_pseudotimes.csv')
        return z_df

In [ ]:
z_df = run_ouroboros(data, data_type, species = 'mouse', outdir = '.')


In [ ]:
ref_embed =  pd.read_csv('/projects/steiflab/research/hmacdonald/total_RNA/Ouroboros_pypi/Ouroboros/data/reference_embeddings.csv')
# set cell id to be index
ref_embed = ref_embed.set_index('cell_id')

In [ ]:
plot_sphere(z_df, colour_by = 'KNN_phase', palette = None, ref = ref_embed, velocity = None, marker_size = 2, cycle_pole = reference_CC_pole_point, savefig = None, show = True)

In [ ]:
plot_sphere(z_df, colour_by = 'cell_cycle_pseudotime', palette = None, ref = ref_embed, velocity = None, marker_size = 2, cycle_pole = reference_CC_pole_point, savefig = None, show = True)

In [ ]:
print(z_df)

In [ ]:
print(z_df)

In [ ]:
z_df.dormancy_depth.unique()

In [ ]:
plot_sphere(z_df, colour_by = 'dormancy_depth', palette = None, ref = ref_embed, velocity = None, marker_size = 2, cycle_pole = reference_CC_pole_point, savefig = None, show = True)

In [ ]:
mtx = z_df[['dim1', 'dim2', 'dim3']].values
radius = np.mean(np.linalg.norm(mtx, axis=1)) - 0.01
offset = 0.01 * radius  # Slight offset above the sphere

In [ ]:
color_vals = z_df['dormancy_depth']
x = z_df['dim1'].values
y = z_df['dim2'].values
z = z_df['dim3'].values

In [ ]:
# Project ALL points first
x_proj, y_proj, z_proj = project_above_sphere(x, y, z, radius, offset)

# Create NaN mask using the index from z_df
mask_valid = color_vals.notna().values
mask_na = ~mask_valid

In [ ]:
color_array = color_vals.fillna(0).to_numpy()  # dummy values for color


In [ ]:
traces = []

In [ ]:

# Plot valid points with color axis
if mask_valid.any():
    traces.append(go.Scatter3d(
        x=x_proj[mask_valid], y=y_proj[mask_valid], z=z_proj[mask_valid],
        mode='markers',
        marker=dict(size=marker_size, color=color_array[mask_valid], coloraxis='coloraxis'),
        name=color_by,
        showlegend=False
    ))

In [ ]:
traces

In [ ]:
# Plot missing values in grey
if mask_na.any():
    traces.append(go.Scatter3d(
        x=x_proj[mask_na], y=y_proj[mask_na], z=z_proj[mask_na],
        mode='markers',
        marker=dict(size=5, color='lightgrey'),
        name='NA',
        showlegend=True
    ))

In [ ]:
traces

In [ ]:
 # Extract coordinates and color values
color_vals = z_df[color_by]
x = z_df['dim1'].values
y = z_df['dim2'].values
z = z_df['dim3'].values

# Project ALL points first
x_proj, y_proj, z_proj = project_above_sphere(x, y, z, radius, offset)

# Create NaN mask using the index from z_df
mask_valid = color_vals.notna().values
mask_na = ~mask_valid

color_array = color_vals.fillna(0).to_numpy()  # dummy values for color

# Plot valid points with color axis
if mask_valid.any():
    traces.append(go.Scatter3d(
        x=x_proj[mask_valid], y=y_proj[mask_valid], z=z_proj[mask_valid],
        mode='markers',
        marker=dict(size=marker_size, color=color_array[mask_valid], coloraxis='coloraxis'),
        name=color_by,
        showlegend=False
    ))

# Plot missing values in grey
if mask_na.any():
    traces.append(go.Scatter3d(
        x=x_proj[mask_na], y=y_proj[mask_na], z=z_proj[mask_na],
        mode='markers',
        marker=dict(size=marker_size, color='lightgrey'),
        name='NA',
        showlegend=True
    ))

In [ ]:
is_cont = is_continuous(z_df['dormancy_depth'])
is_cont

In [ ]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import anndata as ad
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from matplotlib import cm
from matplotlib.colors import Normalize

from matplotlib.cm import get_cmap
from plotly.io import write_image
import seaborn as sns


reference_CC_pole_point = [0.86202236, 0.24824865, 0.44191636]

phase_pal_transition = {
    'G1':'#1f77b4',
       'S': '#ff7f0e',
       'G2M': '#2ca02c', 
       'G0': 'black', 
       'G1-G0 transition': '#d62728'}


def is_continuous(series):
    return pd.api.types.is_numeric_dtype(series) and series.notna().sum() > 10


def make_sphere_surface(radius, points=300):
    phi = np.linspace(0, np.pi, points)
    theta = np.linspace(0, 2 * np.pi, points)
    phi, theta = np.meshgrid(phi, theta)
    x = radius * np.sin(phi) * np.cos(theta)
    y = radius * np.sin(phi) * np.sin(theta)
    z = radius * np.cos(phi)
    return go.Surface(x=x, y=y, z=z, colorscale=[[0, 'lightgray'], [1, 'lightgray']],
                      opacity=1, showscale=False, name='', legendgroup='none',
                      contours=dict(x=dict(show=False), y=dict(show=False), z=dict(show=False)))

def project_above_sphere(x, y, z, radius, offset):
    points = np.stack([x, y, z], axis=1)
    normals = points / np.linalg.norm(points, axis=1, keepdims=True)
    projected_points = normals * (radius + offset)
    return projected_points[:, 0], projected_points[:, 1], projected_points[:, 2]


def make_pole_trace(pole, name, color, width, radius, extension=1.3):
    point = np.array(pole)
    point_norm = point / np.linalg.norm(point) * radius
    opp = -point_norm
    return go.Scatter3d(
        x=[point_norm[0]*extension, opp[0]*extension],
        y=[point_norm[1]*extension, opp[1]*extension],
        z=[point_norm[2]*extension, opp[2]*extension],
        mode='lines',
        line=dict(color=color, width=width),
        name=name
    )


def make_reference_traces(ref, radius, offset, marker_size=5, alpha=0.07):
    traces = []
    ref_pal = phase_pal_transition
    for phase, color in ref_pal.items():
        df = ref[ref['phase'] == phase]
        x, y, z = df['dim1'].values, df['dim2'].values, df['dim3'].values
        x, y, z = project_above_sphere(x, y, z, radius, offset)
        traces.append(go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(size=marker_size, color=color, opacity=alpha),
                                   name=f"Reference {phase}", showlegend=True))
    return traces





def make_new_traces(z_df, radius, offset, color_by='KNN_phase', palette=None, marker_size=3, is_continuous=False):
    traces = []
    if is_continuous:
        # Extract coordinates and color values
        color_vals = z_df[color_by]
        x = z_df['dim1'].values
        y = z_df['dim2'].values
        z = z_df['dim3'].values

        # Project ALL points first
        x_proj, y_proj, z_proj = project_above_sphere(x, y, z, radius, offset)

        # Create NaN mask using the index from z_df
        mask_valid = color_vals.notna().values
        mask_na = ~mask_valid

        color_array = color_vals.fillna(0).to_numpy()  # dummy values for color

        # Plot valid points with color axis
        if mask_valid.any():
            traces.append(go.Scatter3d(
                x=x_proj[mask_valid], y=y_proj[mask_valid], z=z_proj[mask_valid],
                mode='markers',
                marker=dict(size=marker_size, color=color_array[mask_valid], coloraxis='coloraxis'),
                name=color_by,
                showlegend=False
            ))

        # Plot missing values in grey
        if mask_na.any():
            traces.append(go.Scatter3d(
                x=x_proj[mask_na], y=y_proj[mask_na], z=z_proj[mask_na],
                mode='markers',
                marker=dict(size=marker_size, color='grey'),
                name='NA',
                showlegend=True
            ))

    else:
        if palette is None:
            palette = {}
        for phase, color in palette.items():
            phase_df = z_df[z_df[color_by] == phase]
            x, y, z = phase_df['dim1'].values, phase_df['dim2'].values, phase_df['dim3'].values
            x, y, z = project_above_sphere(x, y, z, radius, offset)
            traces.append(go.Scatter3d(x=x, y=y, z=z, mode='markers',
                                       marker=dict(size=marker_size, color=color),
                                       name=phase))
    return traces




def make_velocity_vectors(z_df, velocity):
    vel = velocity[['dim1', 'dim2', 'dim3']].to_numpy()
    umap = z_df[['dim1', 'dim2', 'dim3']].to_numpy()
    arrows = [
        go.Scatter3d(x=[u[0], u[0] + v[0]], y=[u[1], u[1] + v[1]], z=[u[2], u[2] + v[2]],
                     mode='lines', line=dict(color='black', width=1), showlegend=False, legendgroup='velocity')
        for u, v in zip(umap, vel)
    ]
    cone = go.Cone(x=umap[:, 0] + vel[:, 0], y=umap[:, 1] + vel[:, 1], z=umap[:, 2] + vel[:, 2],
                   u=vel[:, 0], v=vel[:, 1], w=vel[:, 2], sizemode="scaled", sizeref=0.5, anchor="tail",
                   colorscale=[[0, "black"], [1, "black"]], showscale=False, name='Velocity Vectors', showlegend=True,
                   legendgroup='velocity')
    return arrows, cone



def seaborn_to_plotly(palette_name, n_colors=256):
    cmap = cm.get_cmap(palette_name)  # use matplotlib for everything
    colors = [cmap(i / (n_colors - 1))[:3] for i in range(n_colors)]
    return [[i / (n_colors - 1), f"rgb({r*255:.0f},{g*255:.0f},{b*255:.0f})"] for i, (r, g, b) in enumerate(colors)]

def normalize_colormap(cmap_name='mako', vmin=-1, vmax=0, n_colors=256):
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.get_cmap(cmap_name)
    colors = [cmap(norm(np.linspace(vmin, vmax, n_colors)[i]))[:3] for i in range(n_colors)]
    return [[i / (n_colors - 1), f"rgb({r*255:.0f},{g*255:.0f},{b*255:.0f})"] for i, (r, g, b) in enumerate(colors)]



def plot_sphere(z_df, colour_by = 'KNN_phase', palette = None, ref = None, velocity = None, marker_size = 2, cycle_pole = reference_CC_pole_point, savefig = None, show = False):
    fig_data = []

    # Sphere properties
    mtx = z_df[['dim1', 'dim2', 'dim3']].values
    radius = np.mean(np.linalg.norm(mtx, axis=1)) - 0.01
    offset = 0.01 * radius  # Slight offset above the sphere

    # Make grey surface of sphere
    sphere = make_sphere_surface(radius)
    fig_data.append(sphere)
  
    #fig_data.append(make_pole_trace(cycle_pole, 'Cell cycle pole', color='grey', width=25, radius = radius, extension=1.3))

    if ref is not None:
        ref_traces = make_reference_traces(ref, radius, offset, marker_size=5, alpha=0.07)
        fig_data += ref_traces

    if colour_by in ['dormancy_depth', 'cell_cycle_pseudotime']:
        is_cont = True
    else:
        is_cont = is_continuous(z_df[colour_by])

    if is_cont:
        if isinstance(palette, str):
            palette = seaborn_to_plotly(palette)
        elif palette is None and colour_by == 'cell_cycle_pseudotime':
            palette = normalize_colormap('rocket_r', vmin=0, vmax=1)
        elif palette is None and colour_by == 'dormancy_depth':
            palette = normalize_colormap('mako', vmin=-1, vmax=0)
        elif palette is None:
            palette = seaborn_to_plotly('viridis')
        
    else: 
        if colour_by == 'KNN_phase':
            palette = phase_pal_transition
        elif colour_by != 'KNN_phase' and palette is None:
            # Generate a palette if none is provided
            unique_labels = z_df[colour_by].unique()
            cmap = get_cmap('tab20')
            palette = {
                label: '#{:02x}{:02x}{:02x}'.format(
                    int(r * 255), int(g * 255), int(b * 255)
                )
                for i, label in enumerate(unique_labels)
                for r, g, b, _ in [cmap(i / max(len(unique_labels) - 1, 1))]
            }
        else:
            palette = palette 


    scatter_traces = make_new_traces(z_df, radius, offset, color_by=colour_by, palette=palette, marker_size=marker_size, is_continuous=is_cont)
    fig_data += scatter_traces

    if velocity is not None:
        arrow_traces, cone_trace = make_velocity_vectors(z_df, velocity)
        fig_data += arrow_traces + [cone_trace]
        
    fig = go.Figure(data=fig_data)
    
    fig.update_layout(
        margin=dict(l=5, r=5, t=5, b=5),
        scene=dict(
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, visible=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, visible=False),
            zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, visible=False)
        ),
        legend=dict(
            x=0.9, 
            y=0.3, 
            font=dict(size=14),
            itemsizing='constant',  
        ),
        showlegend=True,
    )
    
    if is_cont:
        fig.update_layout(
            coloraxis=dict(
                colorscale=palette,
                colorbar=dict(
                    title=dict(
                        text=colour_by.replace('_', ' ').capitalize(),
                        side="top",
                        font=dict(size=16)
                    ),
                    len=0.4,
                    thickness=20,
                    x=0.8,
                    y=0.5,
                    yanchor="middle"
                )
            )
        )
    if savefig is not None: 
        fig.write_html(savefig)
    if show == True:
        fig.show()

